# 06 — Visualisierungen der Cross-Gemeinde-Evaluation

Dieses Notebook erzeugt **5 Plots** fuer die Projektarbeit:

1. **Heatmap**: MAE pro Modell x Gemeinde (20 x 18 Matrix)
2. **Balkendiagramm**: Mittlere MAE pro Modell (Ranking)
3. **Bekannt vs. Unbekannt**: Generalisierungsluecke pro Modell
4. **Modelltyp x Datenvariante**: Gruppierter Vergleich
5. **Trainingszeiten**: Laufzeit pro Pipeline-Skript

Alle Plots werden als PNG (300 DPI) und PDF gespeichert.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch

In [ ]:
SCRIPT_DIR = Path(".").resolve().parent
OUTPUT_DIR = SCRIPT_DIR / "ergebnisse"
FIG_DIR = SCRIPT_DIR.parent / "Unterlagen" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Matplotlib-Einstellungen fuer Druckqualitaet
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
})

In [ ]:
# Daten laden
df = pd.read_csv(OUTPUT_DIR / "cross_evaluation.csv")

# Modelle nach mittlerem MAE sortieren (bestes zuerst)
model_order = df.groupby("modell")["MAE"].mean().sort_values().index.tolist()
gemeinde_order = sorted(df["gemeinde"].unique())

# Bekannte Gemeinden pro Datenvariante
known_sets = {
    "D1_single": {"GM0047"},
    "D2_multi4": {"GM0047", "GM0059", "GM0281", "GM0590"},
    "D3_mittel6": {"GM0047", "GM0312", "GM0590", "GM0546", "GM0629", "GM1681"},
    "D4_gross10": {"GM0047", "GM0312", "GM0590", "GM0546", "GM0629", "GM1681",
                   "GM0281", "GM1950", "GM1969", "GM1930"},
    "D5_fremd": {"GM0312", "GM0590", "GM0546", "GM0629", "GM1681"},
}

print(f"Datensatz: {len(df)} Zeilen ({df['modell'].nunique()} Modelle x {df['gemeinde'].nunique()} Gemeinden)")

## Plot 1: Heatmap — MAE pro Modell x Gemeinde

Die Heatmap zeigt die vollstaendige 20x18-Ergebnismatrix. Farbskala: Gruen (niedrig/gut) bis Rot (hoch/schlecht). Die Zeilen sind nach mittlerem MAE sortiert (bestes Modell oben).

In [ ]:
pivot = df.pivot(index="modell", columns="gemeinde", values="MAE")
pivot = pivot.reindex(index=model_order, columns=gemeinde_order)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(
    pivot, annot=True, fmt=".0f", cmap="RdYlGn_r",
    vmin=5, vmax=100, linewidths=0.5, ax=ax,
    cbar_kws={"label": "MAE (s)"},
)
ax.set_title("Cross-Gemeinde-Evaluation: MAE (s) pro Modell und Gemeinde")
ax.set_ylabel("Modell (sortiert nach Ø MAE)")
ax.set_xlabel("Gemeinde")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
fig.savefig(FIG_DIR / "cross_eval_heatmap.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "cross_eval_heatmap.pdf", bbox_inches="tight")
plt.show()

## Plot 2: Balkendiagramm — Mittlere MAE pro Modell

Horizontales Ranking aller 20 Modelle. Farbkodierung nach Modelltyp:
- Rot: XGBoost
- Gruen: Random Forest
- Blau: MLPv2
- Orange: Ridge

In [ ]:
means = df.groupby("modell")["MAE"].mean().reindex(model_order)

colors = []
for m in model_order:
    if m.startswith("XGB"): colors.append("#e74c3c")
    elif m.startswith("RF"): colors.append("#2ecc71")
    elif m.startswith("MLPv2"): colors.append("#3498db")
    elif m.startswith("Ridge"): colors.append("#f39c12")
    else: colors.append("#95a5a6")

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(model_order)), means.values, color=colors)
ax.set_yticks(range(len(model_order)))
ax.set_yticklabels(model_order)
ax.set_xlabel("Mittlere MAE ueber 18 Gemeinden (s)")
ax.set_title("Cross-Gemeinde-Evaluation: Mittlere MAE pro Modell")
ax.invert_yaxis()

for bar, val in zip(bars, means.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2, f"{val:.1f}s",
            va="center", ha="left", fontsize=9)

legend_elements = [
    Patch(facecolor="#e74c3c", label="XGBoost"),
    Patch(facecolor="#2ecc71", label="Random Forest"),
    Patch(facecolor="#3498db", label="MLPv2 (Entity Emb.)"),
    Patch(facecolor="#f39c12", label="Ridge Regression"),
]
ax.legend(handles=legend_elements, loc="lower right")
ax.set_xlim(0, max(means.values) * 1.15)

fig.savefig(FIG_DIR / "cross_eval_mean_mae.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "cross_eval_mean_mae.pdf", bbox_inches="tight")
plt.show()

## Plot 3: Bekannt vs. Unbekannt (Generalisierungsluecke)

Fuer jedes Modell wird die MAE separat fuer bekannte (im Training enthaltene) und unbekannte Gemeinden dargestellt. Der Abstand zwischen den Balken ist die **Generalisierungsluecke** — je kleiner, desto besser generalisiert das Modell.

In [ ]:
rows = []
for model_name in model_order:
    dv = None
    for k in known_sets:
        if k in model_name:
            dv = k
            break
    if dv is None:
        continue
    known = known_sets[dv]
    m_df = df[df["modell"] == model_name]
    k_mae = m_df[m_df["gemeinde"].isin(known)]["MAE"].mean()
    u_mae = m_df[~m_df["gemeinde"].isin(known)]["MAE"].mean()
    rows.append({"modell": model_name, "bekannt": k_mae, "unbekannt": u_mae})

bv_df = pd.DataFrame(rows)
x = np.arange(len(bv_df))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width / 2, bv_df["bekannt"], width, label="Bekannte Gemeinden", color="#2ecc71", alpha=0.85)
bars2 = ax.bar(x + width / 2, bv_df["unbekannt"], width, label="Unbekannte Gemeinden", color="#e74c3c", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(bv_df["modell"], rotation=45, ha="right")
ax.set_ylabel("MAE (s)")
ax.set_title("Generalisierung: MAE auf bekannten vs. unbekannten Gemeinden")
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=8)

fig.savefig(FIG_DIR / "cross_eval_known_vs_unknown.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "cross_eval_known_vs_unknown.pdf", bbox_inches="tight")
plt.show()

## Plot 4: Modelltyp x Datenvariante

Gruppierter Vergleich: Fuer jeden der 4 Modelltypen wird gezeigt, wie sich die 5 Datenvarianten (D1–D5) auf die mittlere Cross-Eval-MAE auswirken.

**Erwartung**: Mehr Trainingsdaten (D1→D4) sollten die MAE senken. Tatsaechlich gilt das nur fuer Ridge und MLPv2 — bei Baummodellen verschlechtert sich die Generalisierung!

In [ ]:
df["modelltyp"] = df["modell"].str.split("_").str[0]
df["datenvariante"] = df["modell"].str.replace(r"^[^_]+_", "", regex=True)

type_data_means = df.groupby(["modelltyp", "datenvariante"])["MAE"].mean().reset_index()

type_order = ["Ridge", "MLPv2", "RF", "XGB"]
data_order = ["D1_single", "D2_multi4", "D3_mittel6", "D4_gross10", "D5_fremd"]
data_colors = {"D1_single": "#3498db", "D2_multi4": "#2ecc71", "D3_mittel6": "#f39c12",
               "D4_gross10": "#e74c3c", "D5_fremd": "#9b59b6"}

fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(type_order))
width = 0.15

for i, dv in enumerate(data_order):
    subset = type_data_means[type_data_means["datenvariante"] == dv]
    vals = []
    for t in type_order:
        row = subset[subset["modelltyp"] == t]
        vals.append(row["MAE"].values[0] if len(row) > 0 else 0)
    mask = [v > 0 for v in vals]
    positions = [x_pos[j] + (i - 2) * width for j in range(len(type_order)) if mask[j]]
    heights = [v for v in vals if v > 0]
    ax.bar(positions, heights, width, label=dv, color=data_colors[dv], alpha=0.85)

ax.set_xticks(x_pos)
ax.set_xticklabels(type_order)
ax.set_ylabel("Mittlere MAE ueber 18 Gemeinden (s)")
ax.set_title("Modelltyp x Datenvariante: Mittlere Cross-Eval MAE")
ax.legend(title="Datenvariante")

fig.savefig(FIG_DIR / "cross_eval_type_vs_data.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "cross_eval_type_vs_data.pdf", bbox_inches="tight")
plt.show()

## Plot 5: Trainingszeiten

Horizontales Balkendiagramm der Skript-Laufzeiten. Zeigt, welche Pipeline-Schritte am meisten Rechenzeit benoetigen.

In [ ]:
timing = pd.read_csv(OUTPUT_DIR / "training_times.csv")
timing = timing.sort_values("laufzeit_s", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(timing["skript"], timing["laufzeit_s"], color="#3498db")
ax.set_xlabel("Laufzeit (s)")
ax.set_title("Skript-Laufzeiten")
for bar, val in zip(bars, timing["laufzeit_s"]):
    label = f"{val:.0f}s" if val < 600 else f"{val/60:.0f}min"
    ax.text(val + 10, bar.get_y() + bar.get_height() / 2, label, va="center", fontsize=9)
ax.set_xlim(0, timing["laufzeit_s"].max() * 1.15)

fig.savefig(FIG_DIR / "training_times.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "training_times.pdf", bbox_inches="tight")
plt.show()

print(f"\n5 Plots gespeichert in: {FIG_DIR}")